# 演習 1 ｜ deutsch_implementation.ipynb

**古典 2 回 vs 量子 1 回 — Deutsch アルゴリズムを自分で組む**

---

## このノートブックで確認すること

関数 $f : \{0,1\} \to \{0,1\}$ が **定数** か **バランス** かを判定する Deutsch 問題:

- **古典**: $f(0), f(1)$ の両方を計算 → 必ず **2 回** 呼ぶ
- **量子 (Deutsch)**: 位相キックバックと干渉で **1 回** の $U_f$ 評価で判定

## 所要時間と書く量

- 所要: **5 分**
- **学生が書くのは、セル 4 と セル 5 の TODO (合計 2 つのオラクル、各 1-2 行)**
- セル 1, 2, 3, 6, 7 は「実行するだけ」。内容は読んで理解してください

## 流れ

1. セル 1-2: 古典版の call_count を確認 (常に 2)
2. セル 3: Deutsch 回路の枠組みを理解
3. セル 4-5: ★ オラクル 2 つを書く ★
4. セル 6-7: 4 種すべて実行して古典と比較
5. セル 8: 自分の言葉で観察をまとめる

---


---
## 環境セットアップ (Google Colab の場合は毎回実行)

第1回ガイダンスの Setup ノートブックと同じ手順です。
ローカル環境 (VSCode 等) で既に Qiskit をインストール済みの場合はスキップしてOK。


In [ ]:
# Qiskit のインストール (1-2 分かかります)
!pip install -q qiskit[visualization] qiskit-aer

# バージョン確認
import qiskit
print(f"Qiskit version: {qiskit.__version__}")
print("インストール完了")

## セル 1 ｜【提供・実行のみ】 セットアップ


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

SHOTS = 1000
sim = AerSimulator()

print("セットアップ完了")


## セル 2 ｜【提供・実行のみ】 古典版の Deutsch

`CallCounter` でブラックボックス関数 $f$ の呼び出し回数を記録します。
4 種類の $f$ で実行して、**常に 2 回呼ばれる** ことを確認してください。


In [ ]:
class CallCounter:
    """関数呼び出し回数を数えるラッパー"""
    def __init__(self, f):
        self.f = f
        self.count = 0
    def __call__(self, x):
        self.count += 1
        return self.f(x)


def classical_deutsch(f):
    """古典版: f(0) と f(1) を調べて定数/バランスを判定"""
    y0 = f(0)
    y1 = f(1)
    return "constant" if y0 == y1 else "balanced"


# 4 種類の関数
functions = {
    "f₀ (f=0)":  lambda x: 0,
    "f₁ (f=1)":  lambda x: 1,
    "f₂ (f=x)":  lambda x: x,
    "f₃ (f=¬x)": lambda x: 1 - x,
}

print(f"{'関数':<12} {'判定':<12} {'古典call_count'}")
print("─" * 40)
for name, f in functions.items():
    counted = CallCounter(f)
    result = classical_deutsch(counted)
    print(f"{name:<12} {result:<12} {counted.count}")


## セル 3 ｜【提供・実行のみ】 Deutsch 量子回路の枠組み

Deutsch 回路は以下の 5 ステップ:

1. `|0⟩|1⟩` に初期化
2. 両方の qubit に $H$ → `|+⟩|-⟩` の積状態
3. **オラクル $U_f$** を適用 ← ここが $f$ によって変わる
4. `qubit 0` に $H$
5. `qubit 0` を測定 → **0 なら定数、1 ならバランス**

`deutsch_circuit(oracle)` は oracle を引数に取って完全な回路を返します。
オラクル自体は次のセルで作ります。


In [ ]:
def deutsch_circuit(oracle):
    """共通 Deutsch 回路。oracle を引数に取って完全な回路を返す"""
    qc = QuantumCircuit(2, 1)
    qc.x(1)                          # qubit 1 → |1⟩
    qc.h([0, 1])                     # → |+⟩|-⟩ の積状態
    qc.compose(oracle, inplace=True) # U_f を合成
    qc.h(0)                          # qubit 0 に H
    qc.measure(0, 0)                 # qubit 0 のみ測定
    return qc


# 動作確認: f₀ (常に 0) のオラクルは「何もしない」
oracle_f0 = QuantumCircuit(2)    # ゲートを1つも置かない = 何もしない oracle

qc_f0 = deutsch_circuit(oracle_f0)
counts_f0 = sim.run(qc_f0, shots=SHOTS).result().get_counts()
print("f₀ (定数 0) の判定結果:", counts_f0, "← 0 が出れば constant")


## セル 4 ｜【★ 演習 1 ★】 バランス関数 f₂ (f(x) = x) のオラクルを書く

$f_2(x) = x$ は「入力 $x$ をそのまま返す」バランス関数。

$U_{f_2} |x\rangle |y\rangle = |x\rangle |y \oplus f_2(x)\rangle = |x\rangle |y \oplus x\rangle$

これは何ゲートで実装できる? (ヒント: 講義スライドで見た通り)

### 予想を書く (30 秒)

- バランス関数なので、Deutsch で測定すると **0** か **1** か?
  - あなたの予想: **________**

### あなたのタスク

下の `# TODO` にオラクルを書き加えてください (**1 行**):


In [ ]:
# f₂ (f(x) = x) の oracle を構築する
oracle_f2 = QuantumCircuit(2)

# ─────────────────────────────────────────
# TODO: f(x) = x を表すゲートを oracle_f2 に追加する (1 行)
# ヒント: qubit 0 を制御、qubit 1 をターゲットとする CNOT
#         → oracle_f2.cx(?, ?)


# ─────────────────────────────────────────

# Deutsch 回路を作成して実行
qc_f2 = deutsch_circuit(oracle_f2)
counts_f2 = sim.run(qc_f2, shots=SHOTS).result().get_counts()
print("f₂ (バランス) の判定結果:", counts_f2)
print("→ 1 が出れば balanced ✓")


## セル 5 ｜【★ 演習 2 ★】 バランス関数 f₃ (f(x) = ¬x) のオラクルを書く

$f_3(x) = \lnot x$ は「入力を反転して返す」バランス関数。

$U_{f_3} |x\rangle |y\rangle = |x\rangle |y \oplus f_3(x)\rangle = |x\rangle |y \oplus \lnot x\rangle$

$y \oplus \lnot x = y \oplus (1 \oplus x) = (y \oplus 1) \oplus x$

つまり:

$f_3(x) = \lnot x$ を実装するには、以下の手順でゲートを並べます:

1. 標的 qubit に **X**
2. 制御 → 標的に **CNOT**

別手順として「先に CNOT、後に X」でも同じ結果が得られます。ただし **一般にゲートの順序は入れ替えられません** (量子ゲートは行列の積なので非可換)。
この例が入れ替え可能なのは偶然ではなく、X と CNOT の特別な関係によるものです。考察問 2 で理由を考えてみてください。

### 予想を書く (30 秒)

- バランス関数なので、Deutsch で測定すると **0** か **1** か?
  - あなたの予想: **___**


In [ ]:
# f₃ (f(x) = ¬x) の oracle を構築する
oracle_f3 = QuantumCircuit(2)

# ─────────────────────────────────────────
# TODO: f(x) = ¬x を表すゲートを oracle_f3 に追加する (2 行)
# ヒント: X を qubit 1 にかけてから CNOT(0, 1) が分かりやすい
#         (順序を変えた CNOT(0, 1); X on qubit 1 でも同じ動作。余裕があれば確認)


# ─────────────────────────────────────────

# Deutsch 回路を作成して実行
qc_f3 = deutsch_circuit(oracle_f3)
counts_f3 = sim.run(qc_f3, shots=SHOTS).result().get_counts()
print("f₃ (バランス) の判定結果:", counts_f3)
print("→ 1 が出れば balanced ✓")


## セル 6 ｜【提供・実行のみ】 4 種類の oracle をまとめて実行

残りの定数オラクル f₀, f₁ を用意して、4 種類すべてを実行します。

- **f₀ (常に 0)**: 何もしない oracle (セル 3 で作成済み)
- **f₁ (常に 1)**: qubit 1 に $X$ を 1 回かける (常に $y$ を反転 = 常に 1 を返す)
- **f₂ (= x)**: あなたが書いた (セル 4)
- **f₃ (= ¬x)**: あなたが書いた (セル 5)


In [ ]:
# f₁ (常に 1 を返す) のオラクル
oracle_f1 = QuantumCircuit(2)
oracle_f1.x(1)

# 4 つのオラクルを辞書にまとめる
oracles = {
    "f₀ (f=0)":  ("定数",   oracle_f0),
    "f₁ (f=1)":  ("定数",   oracle_f1),
    "f₂ (f=x)":  ("バランス", oracle_f2),
    "f₃ (f=¬x)": ("バランス", oracle_f3),
}

# 全部実行して測定結果を見る
print(f"{'関数':<12} {'f 種類':<8} {'測定':<8}")
print("─" * 35)
for name, (kind, oracle) in oracles.items():
    qc = deutsch_circuit(oracle)
    counts = sim.run(qc, shots=SHOTS).result().get_counts()
    measured = max(counts, key=counts.get)
    print(f"{name:<12} {kind:<8} {measured:<8}")


## セル 7 ｜【提供・実行のみ】 古典版と量子版のクエリ回数比較

$f$ の呼び出し回数 (= クエリ複雑度) を直接比較します。


In [ ]:
# 古典と量子の比較表
functions_map = {
    "f₀ (f=0)":  lambda x: 0,
    "f₁ (f=1)":  lambda x: 1,
    "f₂ (f=x)":  lambda x: x,
    "f₃ (f=¬x)": lambda x: 1 - x,
}

print(f"{'関数':<12} {'f種類':<8} {'測定':<6} {'古典回数':<10} {'量子回数':<10} {'判定'}")
print("─" * 60)
for (name, (kind, oracle)), f in zip(oracles.items(), functions_map.values()):
    # 量子側
    qc = deutsch_circuit(oracle)
    counts = sim.run(qc, shots=SHOTS).result().get_counts()
    measured = max(counts, key=counts.get)
    verdict = "constant" if measured == "0" else "balanced"
    correct = "✓" if verdict == ("constant" if kind == "定数" else "balanced") else "✗"
    # 古典側
    counted = CallCounter(f)
    classical_deutsch(counted)
    print(f"{name:<12} {kind:<8} {measured:<6} {counted.count:<10} 1{'':<9} {correct} {verdict}")

print()
print("→ どの oracle でも、量子版は 1 回の U_f 呼び出しで判定成功")
print("→ 古典版は必ず 2 回 f を呼ぶ必要がある")


## セル 8 ｜【考察】 自分の言葉でまとめる

以下の質問に日本語で答えてください (このセルを編集)。
AI に聞いても構いませんが、**AI の答えを自分の言葉で言い換えてから** 書いてください。

---

### 1. f₂ のオラクルは CNOT 1 個で書けた。なぜそれで $y \oplus x$ を実現できるのか?

CNOT の真理値表と $y \oplus x$ の関係から説明してください。

**あなたの説明**: _______________________________________

---

### 2. f₃ のオラクルで、`X + CNOT` の順序を入れ替え (CNOT + X) ても、同じ $f_3$ を実現できる。なぜか?

余裕があれば、実際にコードで確かめてから説明してください。

**あなたの説明**: _______________________________________

---

### 3. 古典版は必ず 2 回呼ばれるのに、量子版は 1 回で済む。これはなぜか?

スライドで学んだ「位相キックバック」「干渉」を使って説明してください。

**あなたの説明**: _______________________________________

---

### 4. このアルゴリズムの「定数倍差 (古典 2 → 量子 1)」に対する感想、また Grover や Shor で見せられた「漸近的加速」との違いについてのコメント (自由記述)

**あなたの感想**: _______________________________________

---

> 🎯 この課題の本質は「オラクルを 2 つ書く」ことではなく、「なぜその回路で $f$ を表現できるか」を自分の言葉で説明できること。


## 📌 段階的ヒント (詰まったら開く)

> ヒントは順番に見ること。L3 まで見てから自分で書くのもアリ (ただし必ずセル 4-5 で予想を書いてから)。

### f₂ (f(x) = x) のヒント

**L1** (ゆるいヒント):
$y \oplus f(x)$ の形を思い出す。$f(x) = x$ なら $y \oplus x$ になる。XOR を実現するのは何ゲート?

**L2** (もう少し):
$f(x) = x$ のとき $U_f |x\rangle |y\rangle = |x\rangle |y \oplus x\rangle$。
これは「$x$ を制御、$y$ をターゲットとする CNOT」と等価。

**L3** (ほぼ答え):
```python
oracle_f2.cx(0, 1)
```

### f₃ (f(x) = ¬x) のヒント

**L1**:
$\lnot x = 1 \oplus x$ なので、$y \oplus \lnot x = y \oplus 1 \oplus x = (y \oplus 1) \oplus x$。
先に $y$ を反転させてから f₂ と同じ操作をすれば良い。

**L2**:
$y$ を反転するには X ゲートを qubit 1 にかける。
その後 $x \oplus y$ は CNOT。

**L3**:
```python
oracle_f3.x(1)
oracle_f3.cx(0, 1)
```
または
```python
oracle_f3.cx(0, 1)
oracle_f3.x(1)
```
(どちらでも同じ $f_3$ を実現する)


## 📚 持ち帰り課題 (任意、所要 15-20 分)

### 課題: Deutsch-Jozsa (n-bit 版) への拡張

$n$ ビット版の問題:

- 関数 $f : \{0,1\}^n \to \{0,1\}$ がブラックボックスで与えられる
- $f$ は **定数** (すべての入力に対して同じ値) か、**バランス** (半分が 0、半分が 1) のどちらかと保証されている
- $f$ が定数かバランスかを判定する

### クエリ複雑度の比較

| 方式 | 最悪クエリ回数 |
|:-|:-:|
| 古典 | $2^{n-1} + 1$ |
| 量子 (Deutsch-Jozsa) | **1** |

### ヒント

- 回路: $n$ 個の入力 qubit + 1 個のアンシラ qubit (ターゲット)
- 初期化: 入力は $|0\rangle^{\otimes n}$、アンシラは $|1\rangle$
- 両方に $H$ → 入力は $|+\rangle^{\otimes n}$ の重ね合わせ、アンシラは $|-\rangle$
- $U_f$ 1 回かける
- 入力 qubit 全体に $H^{\otimes n}$
- 入力を測定 → **全て 0 なら定数、1 つでも 1 があればバランス**

### 実装の雛形


In [ ]:
# Deutsch-Jozsa (n-bit 版) の雛形

def deutsch_jozsa_circuit(n, oracle):
    """n ビット Deutsch-Jozsa 回路"""
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)              # アンシラ qubit を |1⟩ に
    qc.h(range(n + 1))   # 全体に H

    qc.compose(oracle, inplace=True)

    qc.h(range(n))       # 入力 qubit に再び H
    qc.measure(range(n), range(n))
    return qc


# 例: n=3 で定数 f(x)=0 のオラクル (何もしない)
n = 3
oracle_const = QuantumCircuit(n + 1)   # 何もしない

qc = deutsch_jozsa_circuit(n, oracle_const)
counts = sim.run(qc, shots=SHOTS).result().get_counts()
print(f"n={n}, f=定数(0):", counts)
print("→ 全て 000 が出れば定数 ✓")

# TODO: あなたが書く
# 1. バランス関数のオラクルを作ってみる (例: f(x) = x₀ つまり x の先頭ビット)
#    ヒント: CX(0, n) で実装できる
# 2. 実行して「全て 0 でない」結果が出るか確認
# 3. 古典で同じ問題を解くと何回呼び出しが必要か、自分でコードを書いて比較


### 考察

- $n = 3$ のとき、古典の最悪クエリ回数は何回か? 実際に確かめよ
- $n = 10$ のとき、古典と量子のクエリ回数の比は?
- Deutsch-Jozsa は「最悪クエリ回数」では指数差だが、「平均クエリ回数」で比較するとどうなるか?

### 条件

- **AI 活用可**: ChatGPT, Claude, Copilot など、何でも使ってOK
- ただし **最終的に、自分の言葉で結果を説明できる** こと
- 実行結果のスクリーンショット + 2-3 行の考察を提出
